# Day 4, hands-on 1: the segment summary, worked

Count, median and return rate per segment, built in one pass, with the denominator travelling beside every rate.

Every placeholder is filled with the option the answer key records, and the notebook is executed
from a clean kernel so every output and every check is visible on the page. The line under each
step says why the other three letters fail.

Where this sits in the day, and the steps this notebook walks.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["typical and spread", "the segment summary", "hands-on: the segment summary", "hands-on: repair a broken summary"], lit=2, title="the day's notebooks", show=False),
    kit.flow(["group without knowing the keys", "three things per segment", "rank them", "write the honest sentence"], title="this notebook's steps", show=False),
)

## Setup

Yesterday's profiled output: 44 Kalpa Retail orders, every amount already a number because the six that would not convert were rejected.

In [2]:
import statistics

orders = kit.load_csv("C2_W01_D04_profiled_STUDENT.csv")
for r in orders:
    r["amount"] = int(r["amount"])

print(len(orders), "profiled orders read from ../data/")
print(sorted({r["segment"] for r in orders}))

44 profiled orders read from ../data/
['Business', 'Retail-Core', 'Retail-Plus', 'Student']


## Step 1. Group without knowing the keys in advance

A dictionary you typed out by hand raises the first time the file holds a segment you did not remember. Build the key the first time you see it instead.

In [3]:
kit.flow(["group without knowing the keys", "three things per segment", "rank them", "write the honest sentence"], lit=0)

In [4]:
# TODO 1. What guards the first sight of a new segment?
#   a) key not in buckets
#   b) buckets[key] is None
#   c) len(buckets) == 0
#   d) key in buckets
buckets = {}
for r in orders:
    key = r["segment"]
    if key not in buckets:
        buckets[key] = {"count": 0, "amounts": [], "returned": 0}
    buckets[key]["count"] += 1
    buckets[key]["amounts"].append(r["amount"])
    if r["status"] == "returned":
        buckets[key]["returned"] += 1

print(sorted(buckets))

['Business', 'Retail-Core', 'Retail-Plus', 'Student']


In [5]:
kit.check("four segments were discovered by reading the file", len(buckets) == 4)
kit.check("Student is among them", "Student" in buckets)
kit.check("the counts add back to 44", sum(b["count"] for b in buckets.values()) == 44)

Indexing a key that does not exist raises before the comparison can run. An empty-dictionary test only ever fires once. The inverted condition creates the bucket for keys that already have one and never for the new ones.

## Step 2. Three things per segment, in one pass

Count, a typical amount and a rate. The typical amount is the decision here, and notebook 1 has already settled which one.

In [6]:
kit.flow(["group without knowing the keys", "three things per segment", "rank them", "write the honest sentence"], lit=1)

In [7]:
# TODO 2. Which statistic goes in the typical column?
#   a) sum(b["amounts"]) / b["count"]
#   b) max(b["amounts"])
#   c) statistics.median(b["amounts"])
#   d) min(b["amounts"])
summary = {}
for key, b in buckets.items():
    summary[key] = {
        "count": b["count"],
        "typical": statistics.median(b["amounts"]),
        "returned": b["returned"],
        "rate": b["returned"] / b["count"],
    }

for key in sorted(summary):
    s = summary[key]
    print(f"{key:<12} n={s['count']:>3}  typical Rs {s['typical']:>7,.0f}  "
          f"returned {s['returned']:>2}  rate {100 * s['rate']:>5.1f}%")

Business     n=  9  typical Rs   2,050  returned  1  rate  11.1%
Retail-Core  n= 14  typical Rs   1,910  returned  5  rate  35.7%
Retail-Plus  n= 11  typical Rs   1,435  returned  4  rate  36.4%
Student      n= 10  typical Rs   1,430  returned  2  rate  20.0%


In [8]:
kit.check("four segments summarised", len(summary) == 4)
kit.check("the returned counts add back to 12",
          sum(s["returned"] for s in summary.values()) == 12)
kit.check("Retail-Core's typical order is about Rs 1,910",
          abs(summary["Retail-Core"]["typical"] - 1910) < 50,
          f'Rs {summary["Retail-Core"]["typical"]:,.0f}')

The mean is the statistic KR4232 owns, and Retail-Core is the segment it sits in, so the mean would report a typical Retail-Core order in the tens of thousands. The maximum and the minimum each describe exactly one order.

## Step 3. Rank them, and look at what the ranking measures

Sort by return rate, best first. Then put the count beside each rate and read the two columns together.

In [9]:
kit.flow(["group without knowing the keys", "three things per segment", "rank them", "write the honest sentence"], lit=2)

In [10]:
# TODO 3. What do you sort on for best first?
#   a) pair[1]["count"]
#   b) pair[1]["rate"]
#   c) -pair[1]["rate"]
#   d) pair[0]
ranked = sorted(summary.items(), key=lambda pair: pair[1]["rate"])

for key, s in ranked:
    print(f"{key:<12} {100 * s['rate']:>5.1f}%   on {s['count']:>3} orders")

Business      11.1%   on   9 orders
Student       20.0%   on  10 orders
Retail-Core   35.7%   on  14 orders
Retail-Plus   36.4%   on  11 orders


In [11]:
kit.check("Business ranks first on rate", ranked[0][0] == "Business")
kit.check("and it is also the smallest segment",
          ranked[0][1]["count"] == min(s["count"] for s in summary.values()))
kit.check("the two best segments are the two smallest",
          {ranked[0][0], ranked[1][0]} ==
          set(sorted(summary, key=lambda k: summary[k]["count"])[:2]))

Sorting on the count ranks by size rather than by performance. The negated rate ranks worst first. Sorting on the key ranks alphabetically, which is what a table does when nobody chose an order.

## Step 4. Write the honest sentence

The deliverable is not the table. It is the sentence somebody repeats in a meeting, and the sentence has to carry the count.

In [12]:
kit.flow(["group without knowing the keys", "three things per segment", "rank them", "write the honest sentence"], lit=3)

In [13]:
# TODO 4. Which sentence would you send?
#   a) verdict_only
#   b) rate_only
#   c) with_count, which carries the count and the caveat
#   d) gap_only
TRUST_FLOOR = 30

b = summary["Business"]
one_more = 100 * (b["returned"] + 1) / b["count"]

with_count = "Business returned " + str(b["returned"]) + " of " + str(b["count"]) + " orders, which is "
with_count += format(100 * b["rate"], ".1f") + " percent on a base too small to rank on"
rate_only = "Business returns at " + format(100 * b["rate"], ".1f") + " percent"
verdict_only = "Business is our best segment"
gap_only = "Business beat Retail-Plus by 25 points"

sentence = with_count
print(sentence)
print("one more Business return would read", format(one_more, ".1f"), "percent")

Business returned 1 of 9 orders, which is 11.1 percent on a base too small to rank on
one more Business return would read 22.2 percent


In [14]:
kit.check("your sentence carries the count", "9" in sentence)
kit.check("and it says the base is too small", "too small" in sentence)
kit.check("one more return would move the rate by more than ten points",
          one_more - 100 * b["rate"] > 10, f"{one_more:.1f} percent after one more")
kit.check("every segment in this file is below the trust floor",
          all(s["count"] < TRUST_FLOOR for s in summary.values()))

## What to post

Post one line with the four letters, then the four counts:

```
1a 2c 3b 4c
Retail-Core 14, Retail-Plus 11, Student 10, Business 9
```

Then the one sentence you would send about Business, in your own words.

In [15]:
kit.flow(["group without knowing the keys", "three things per segment", "rank them", "write the honest sentence"], lit=3, title="the notebook, end to end")
kit.check_summary()